# HEOR Multi-Task Learning

Colab-first downstream modeling notebook following the R0/Manuscript3 layout: explicit setup, data preparation, split checks, model-specific training sections, saved predictions, and summary tables.

# 1. Import Libraries, Configuration Setup, and Load Dataset

## 1.1 Install and import libraries

In [ ]:
# Notebook / package setup
import importlib.util
import subprocess
import sys

required = {
    "numpy": "numpy",
    "pandas": "pandas",
    "joblib": "joblib",
    "lightgbm": "lightgbm",
    "sklearn": "scikit-learn",
    "torch": "torch",
    "transformers": "transformers",
    "tqdm": "tqdm",
}

missing = [
    pip_name
    for module_name, pip_name in required.items()
    if importlib.util.find_spec(module_name) is None
]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import gc
import json
import os
import random
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")


## 1.2 Seeds and device

In [ ]:
SEED = 42
RANDOM_STATE = SEED


def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_FP16 = torch.cuda.is_available()

print("Device:", DEVICE)
print("Seed:", SEED)
print("FP16 available:", USE_FP16)


## 1.3 Configuration

In [ ]:
# Portable project-path setup.
# Override any of these by exporting the matching env var before launching
# the notebook (e.g. `export PROJECT_ROOT=/path/to/VulnerableCancerPatients`):
#   PROJECT_ROOT  - root of the experiment tree
#   SCRIPTS_DIR   - shared helper modules (default: PROJECT_ROOT/scripts)
#   DATA_DIR      - shared data directory (default: PROJECT_ROOT/data)
import os
import sys
from pathlib import Path

FOLDER_NAME = "04_HEOR_MTL"
COLAB_DEFAULT = Path("/content/drive/MyDrive/NLP_Projects/VulnerableCancerPatients")


def _resolve_project_root() -> Path:
    env = os.environ.get("PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    try:
        from google.colab import drive  # type: ignore

        drive.mount("/content/drive", force_remount=False)
        if COLAB_DEFAULT.exists():
            return COLAB_DEFAULT.resolve()
    except ImportError:
        pass
    cwd = Path.cwd().resolve()
    if cwd.name == FOLDER_NAME:
        return cwd.parent
    if (cwd / FOLDER_NAME).exists():
        return cwd
    return cwd


PROJECT_ROOT = _resolve_project_root()
BASE_DIR = PROJECT_ROOT / FOLDER_NAME if (PROJECT_ROOT / FOLDER_NAME).exists() else PROJECT_ROOT
SCRIPTS_DIR = Path(os.environ.get("SCRIPTS_DIR", PROJECT_ROOT / "scripts")).expanduser().resolve()
DATA_DIR = Path(os.environ.get("DATA_DIR", PROJECT_ROOT / "data")).expanduser().resolve()
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

print("Project root:", PROJECT_ROOT)
print("Base:", BASE_DIR)
print("Scripts:", SCRIPTS_DIR, "(exists)" if SCRIPTS_DIR.exists() else "(MISSING)")
print("Data:", DATA_DIR, "(exists)" if DATA_DIR.exists() else "(MISSING)")
print("Outputs:", OUTPUT_DIR)


## 1.4 Mount Drive or use local project folder

In [ ]:
# Path setup consolidated into the portable block above.


## 1.5 Load shared data and split

In [ ]:
from data_utils import (
    CANCER_LABELS,
    EMOTION_LABELS_3,
    EMOTION_PROB_COLS_3,
    HEOR_SUBSCALES,
    ROLE_LABELS,
    prepare_annotation_frame,
)
from metrics import DEFAULT_BOOTSTRAP_N
from train_eval import TrainSettings, run_classifier_search, run_heor_r0_mtl_search

df = prepare_annotation_frame(ANNOTATION_PATH, SPLIT_PATH)
train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "val"].copy()
test_df = df[df["split"] == "test"].copy()

print("Prepared shape:", df.shape)
print(df["split"].value_counts().sort_index())
print("Bootstrap helper default:", DEFAULT_BOOTSTRAP_N)
df[["source_row", "split", "human_emotion_3class", "llm_argmax_3class", "ai_high_need_flag"]].head()


# 2. Data Exploration and Master Data Preparation

## 2.1 Master data check

In [ ]:
print("Columns:", len(df.columns))
print("Rows:", len(df))
print()
print("Human 3-class distribution:")
print(df["human_emotion_3class"].value_counts(dropna=False))
print()
print("LLM argmax 3-class distribution:")
print(df["llm_argmax_3class"].value_counts(dropna=False))
print()
print("HEOR subscale summaries:")
display(df[HEOR_SUBSCALES].describe().T)


## 2.2 Canonical split check

In [ ]:
split_summary = (
    df.groupby("split")
    .agg(
        n=("source_row", "size"),
        source_rows=("source_row", "nunique"),
        human_negative=("human_emotion_3class", lambda s: int((s == "NEGATIVE").sum())),
        human_neutral=("human_emotion_3class", lambda s: int((s == "NEUTRAL").sum())),
        human_positive=("human_emotion_3class", lambda s: int((s == "POSITIVE").sum())),
    )
    .reset_index()
)
display(split_summary)

assert set(df["split"].unique()) == {"train", "val", "test"}
assert df["source_row"].is_unique, "Expected one prepared row per source row."

if "posts" in df.columns:
    repeated_text_splits = df.groupby("posts")["split"].nunique()
    n_repeated_text_leaks = int((repeated_text_splits > 1).sum())
    print("Repeated exact-text groups crossing splits:", n_repeated_text_leaks)

master_split_path = OUTPUT_DIR / "master_prepared_split.csv"
df.to_csv(master_split_path, index=False)
print("Saved prepared split:", master_split_path)


## 2.3 Shared helper functions

In [ ]:
def maybe_smoke_sample(frame: pd.DataFrame, per_split: int = SMOKE_PER_SPLIT) -> pd.DataFrame:
    if not SMOKE_TEST:
        return frame.copy()
    return (
        frame.groupby("split", group_keys=False)
        .apply(lambda x: x.sample(min(len(x), per_split), random_state=SEED))
        .reset_index(drop=True)
    )


def make_train_settings(model_name: str = ALBERT_MODEL_NAME, batch_size: int = BATCH_SIZE, n_iter: int = N_ITERATIONS):
    return TrainSettings(
        model_name=model_name,
        max_length=MAX_TOKEN_LENGTH,
        batch_size=batch_size,
        patience=PATIENCE,
        n_iter=n_iter,
        bootstrap_n=N_BOOT,
        seed=SEED,
    )


def append_metric(metrics_list, metric_row):
    metrics_list.append(metric_row)
    display(pd.DataFrame(metrics_list))


df_run = maybe_smoke_sample(df)
print("Run shape:", df_run.shape)
print(df_run["split"].value_counts().sort_index())


# 3. HEOR Multi-Task Conditions

## 3.1 Composite

In [ ]:
settings = make_train_settings()
metric_rows = run_heor_r0_mtl_search(
    df=df_run,
    output_dir=OUTPUT_DIR,
    condition_name="heor_mtl_composite",
    heor_mode="composite",
    include_aux=False,
    settings=settings,
    use_aux_loss_masks=True,
    primary_only_selection=True,
    role_precision_cap=None,
)
display(metric_rows if isinstance(metric_rows, pd.DataFrame) else pd.DataFrame([metric_rows]))


In [ ]:
metric_files = sorted(p for p in OUTPUT_DIR.glob("heor_mtl_*_metrics.csv") if p.name != "heor_mtl_all_metrics.csv")
weight_files = sorted(OUTPUT_DIR.glob("heor_mtl_*_kendall_task_weights.csv"))

if metric_files:
    all_metrics = pd.concat([pd.read_csv(p) for p in metric_files], ignore_index=True)
    all_metrics.to_csv(OUTPUT_DIR / "heor_mtl_all_metrics.csv", index=False)
    display(all_metrics)
else:
    print("No HEOR MTL metric files found yet.")

if weight_files:
    all_weights = pd.concat([pd.read_csv(p) for p in weight_files], ignore_index=True)
    all_weights.to_csv(OUTPUT_DIR / "kendall_task_weights.csv", index=False)
    display(all_weights)
else:
    print("No Kendall task-weight files found yet.")


## 3.2 Composite + Role/Cancer

In [ ]:
settings = make_train_settings()
metric_rows = run_heor_r0_mtl_search(
    df=df_run,
    output_dir=OUTPUT_DIR,
    condition_name="heor_mtl_composite_rc",
    heor_mode="composite",
    include_aux=True,
    settings=settings,
    use_aux_loss_masks=True,
    primary_only_selection=True,
    role_precision_cap=None,
)
display(metric_rows if isinstance(metric_rows, pd.DataFrame) else pd.DataFrame([metric_rows]))


## 3.3 Subscales

In [ ]:
settings = make_train_settings()
metric_rows = run_heor_r0_mtl_search(
    df=df_run,
    output_dir=OUTPUT_DIR,
    condition_name="heor_mtl_subscales",
    heor_mode="subscales",
    include_aux=False,
    settings=settings,
    use_aux_loss_masks=True,
    primary_only_selection=True,
    role_precision_cap=None,
)
display(metric_rows if isinstance(metric_rows, pd.DataFrame) else pd.DataFrame([metric_rows]))


## 3.4 Subscales + Role/Cancer

In [ ]:
settings = make_train_settings()
metric_rows = run_heor_r0_mtl_search(
    df=df_run,
    output_dir=OUTPUT_DIR,
    condition_name="heor_mtl_subscales_rc",
    heor_mode="subscales",
    include_aux=True,
    settings=settings,
    use_aux_loss_masks=True,
    primary_only_selection=True,
    role_precision_cap=None,
)
display(metric_rows if isinstance(metric_rows, pd.DataFrame) else pd.DataFrame([metric_rows]))


In [ ]:
metric_files = sorted(p for p in OUTPUT_DIR.glob("heor_mtl_*_metrics.csv") if p.name != "heor_mtl_all_metrics.csv")
weight_files = sorted(OUTPUT_DIR.glob("heor_mtl_*_kendall_task_weights.csv"))

if metric_files:
    all_metrics = pd.concat([pd.read_csv(p) for p in metric_files], ignore_index=True)
    all_metrics.to_csv(OUTPUT_DIR / "heor_mtl_all_metrics2.csv", index=False)
    display(all_metrics)
else:
    print("No HEOR MTL metric files found yet.")

if weight_files:
    all_weights = pd.concat([pd.read_csv(p) for p in weight_files], ignore_index=True)
    all_weights.to_csv(OUTPUT_DIR / "kendall_task_weights.csv", index=False)
    display(all_weights)
else:
    print("No Kendall task-weight files found yet.")


## 3.5 Subscales + Role/Cancer with role precision cap

In [ ]:
settings = make_train_settings()
metric_rows = run_heor_r0_mtl_search(
    df=df_run,
    output_dir=OUTPUT_DIR,
    condition_name="heor_mtl_subscales_rc_role_cap",
    heor_mode="subscales",
    include_aux=True,
    settings=settings,
    use_aux_loss_masks=True,
    primary_only_selection=True,
    role_precision_cap=1.0,
)
display(metric_rows if isinstance(metric_rows, pd.DataFrame) else pd.DataFrame([metric_rows]))


## 3.6 Optional direct R0 Subscales+RC ablation

In [ ]:
if RUN_R0_SUBSCALES_RC_ABLATION:
    metric_rows = run_heor_r0_mtl_search(
        df=df_run,
        output_dir=OUTPUT_DIR,
        condition_name="heor_mtl_subscales_rc_r0_setup",
        heor_mode="subscales",
        include_aux=True,
        settings=make_train_settings(),
        use_aux_loss_masks=False,
        primary_only_selection=False,
        role_precision_cap=None,
    )
    display(metric_rows if isinstance(metric_rows, pd.DataFrame) else pd.DataFrame([metric_rows]))
else:
    print("Direct R0 Subscales+RC ablation skipped by default.")


## 3.7 Optional RoBERTa encoder robustness sensitivity

In [ ]:
if RUN_ROBERTA_HEOR_SENSITIVITY:
    # Reviewer robustness check: same Subscales+RC setup with a stronger encoder
    # on a stratified 40% subsample to keep GPU cost manageable.
    df_roberta_heor = (
        df_run.groupby("split", group_keys=False)
        .apply(lambda x: x.sample(frac=ROBERTA_HEOR_SUBSAMPLE_FRAC, random_state=SEED))
        .reset_index(drop=True)
    )
    print("RoBERTa HEOR sensitivity shape:", df_roberta_heor.shape)
    print(df_roberta_heor["split"].value_counts().sort_index())

    roberta_heor_settings = make_train_settings(
        model_name="roberta-base",
        batch_size=ROBERTA_HEOR_BATCH_SIZE,
        n_iter=ROBERTA_HEOR_N_ITERATIONS,
    )
    metric_rows = run_heor_r0_mtl_search(
        df=df_roberta_heor,
        output_dir=OUTPUT_DIR,
        condition_name="heor_mtl_subscales_rc_roberta_sensitivity",
        heor_mode="subscales",
        include_aux=True,
        settings=roberta_heor_settings,
        use_aux_loss_masks=True,
        primary_only_selection=True,
        role_precision_cap=None,
    )
    display(metric_rows if isinstance(metric_rows, pd.DataFrame) else pd.DataFrame([metric_rows]))
else:
    print("RoBERTa HEOR sensitivity skipped. Set RUN_ROBERTA_HEOR_SENSITIVITY=True to run it.")


# 4. Save Predictions and Model Comparison

## 4.1 Collect metric and Kendall-weight files

In [ ]:
metric_files = sorted(p for p in OUTPUT_DIR.glob("heor_mtl_*_metrics.csv") if p.name != "heor_mtl_all_metrics.csv")
weight_files = sorted(OUTPUT_DIR.glob("heor_mtl_*_kendall_task_weights.csv"))

if metric_files:
    all_metrics = pd.concat([pd.read_csv(p) for p in metric_files], ignore_index=True)
    all_metrics.to_csv(OUTPUT_DIR / "heor_mtl_all_metrics.csv", index=False)
    display(all_metrics)
else:
    print("No HEOR MTL metric files found yet.")

if weight_files:
    all_weights = pd.concat([pd.read_csv(p) for p in weight_files], ignore_index=True)
    all_weights.to_csv(OUTPUT_DIR / "kendall_task_weights.csv", index=False)
    display(all_weights)
else:
    print("No Kendall task-weight files found yet.")
